# Train per-layer probes + steering smoke

Run on Colab where the activations live. Reads `data/seed_<N>/activations/*.pt` and `data/seed_<N>/trajectories.jsonl`, trains:

- **Classifier A:** L0 vs L2 (sanity check; should succeed).
- **Classifier B:** L1 vs L2 (central probe, the matched-control comparison).

Saves `data/probes/probes_seed{42,43,44}.json` with per-layer AUC + best-layer probe weights for steering. Bundles the small JSON artifacts at the end so you can pull them down to your Mac and run `scripts/05_make_figures.py` locally.

Final cell does the **steering smoke test** : attaches a hook with α=0 and confirms the model produces the same output as without the hook.

## 1. Mount + clone + install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/agent_faithfulness'
REPO_DIR  = '/content/agent_faithfulness'
GITHUB_URL = 'https://github.com/lebretou/agent_faithfulness.git'

import os, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', GITHUB_URL, REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull'])

%cd $REPO_DIR
!pip install -q -r requirements.txt

## 2. Train probes on all 3 seeds

This loads activations into memory once per seed (~150-200 MB each), runs 5-fold CV per layer (~28 layers × 2 classifiers), and writes JSON. Expect ~2-5 minutes per seed.

In [ ]:
!cd {REPO_DIR} && python scripts/03_train_probes.py \
    --root {DRIVE_ROOT}/data \
    --seeds 42 43 44

## 3. Quick summary of probe AUCs

In [ ]:
import sys, json
sys.path.insert(0, REPO_DIR)
from src.probes import load_probe_results

PROBES_DIR = f'{DRIVE_ROOT}/data/probes'
for seed in (42, 43, 44):
    p = f'{PROBES_DIR}/probes_seed{seed}.json'
    if not os.path.exists(p):
        print(f'seed {seed}: missing {p}')
        continue
    rs = load_probe_results(p)
    print(f'\nseed {seed}:')
    for r in rs:
        print(f'  {r.classifier:14s}  n={r.n_samples:4d} '
              f'best_layer={r.best_layer:2d}  best_auc={r.best_auc:.3f}')

## 4. Bundle small artifacts for local download

Bundles `data/probes/*.json` (small, KB) so you can pull them to your Mac and run figure-making locally.

In [ ]:
import subprocess
BUNDLE = '/content/agent_faithfulness_probes_probes.tar.gz'
subprocess.check_call([
    'tar', '-czf', BUNDLE,
    '-C', DRIVE_ROOT,
    'data/probes',
])
print('size (MB):', os.path.getsize(BUNDLE) / 1e6)
from google.colab import files
files.download(BUNDLE)

## 5. Steering smoke test 

Acceptance: hook attaches/detaches cleanly; α = 0 reproduces baseline behavior.

Loads Qwen2.5-7B, picks the best layer for Classifier B from seed 42's probe, and runs:
1. Baseline forward pass on a small prompt.
2. Same forward pass with the steering hook attached at α=0 — output must match exactly.
3. Same forward pass at α=1 — output should differ.
4. After hook removal, the model returns to baseline.

In [ ]:
import torch, numpy as np
from src.agent import load_model
from src.probes import load_probe_results
from src.steering import (
    probe_direction_from_weights, attach_steering_hook, make_steering_hook,
)

model, tok = load_model('Qwen/Qwen2.5-7B-Instruct', dtype='bfloat16')

# Best layer + weights from seed 42's Classifier B (L1 vs L2).
rs = load_probe_results(f'{PROBES_DIR}/probes_seed42.json')
res_b = next(r for r in rs if r.classifier == 'B_L1_vs_L2')
best_layer = res_b.best_layer
direction = probe_direction_from_weights(np.asarray(res_b.best_layer_weights))
print(f'Steering at layer {best_layer}; direction shape={direction.shape}; '
      f'norm={np.linalg.norm(direction):.4f}')

prompt_ids = tok('The capital of France is', return_tensors='pt').input_ids.to(model.device)

with torch.no_grad():
    out_baseline = model(prompt_ids).logits[0, -1].float().cpu().numpy()

# α = 0: must equal baseline.
h = attach_steering_hook(model, best_layer, direction, alpha=0.0)
with torch.no_grad():
    out_alpha0 = model(prompt_ids).logits[0, -1].float().cpu().numpy()
h.remove()

# α = 1: must DIFFER from baseline.
h = attach_steering_hook(model, best_layer, direction, alpha=1.0)
with torch.no_grad():
    out_alpha1 = model(prompt_ids).logits[0, -1].float().cpu().numpy()
h.remove()

# After remove: must equal baseline again.
with torch.no_grad():
    out_post = model(prompt_ids).logits[0, -1].float().cpu().numpy()

max_diff_alpha0 = float(np.max(np.abs(out_baseline - out_alpha0)))
max_diff_alpha1 = float(np.max(np.abs(out_baseline - out_alpha1)))
max_diff_post  = float(np.max(np.abs(out_baseline - out_post)))

print(f'\nmax |Δlogits| α=0:  {max_diff_alpha0:.6f}  (must be ~0)')
print(f'max |Δlogits| α=1:  {max_diff_alpha1:.6f}  (must be > 0.1)')
print(f'max |Δlogits| post: {max_diff_post:.6f}   (must be ~0 after detach)')

assert max_diff_alpha0 < 1e-4, 'α=0 should reproduce baseline exactly'
assert max_diff_alpha1 > 0.05, 'α=1 should produce visible logit shift'
assert max_diff_post  < 1e-4, 'Post-detach logits should match baseline'
print('\n✅ Steering smoke test PASSED')